# Fuse location data

October 2025

- Commercial GPS data is insufficient for race traces. 

- It is only 1hz and not very precise.

- Use  fancy math to fuse the location data into a race trace

- Since the algorithms rely on sequential data points, the data will need to be processed by race 

- Also, since sensor data logging started before the races, will trim the data based on start/finish line location first. Again, by race.

- Sensor data is updated at different frequencies, so first we get the frequencies for each table



## F1 style data fusion

For F1-style telemetry analysis, here are the best practices for fusing multi-frequency sensor data:
### 1. Choose a Master Timeline

Use your highest-priority sensor as the time base (usually Location/GPS at ~1-10 Hz for lap analysis)
For detailed dynamics analysis, use Gyroscope/Accelerometer at 100 Hz

### 2. Resampling Strategies
Upsampling (low → high frequency):
python# Interpolate GPS data to match 100Hz gyro
df_location_upsampled = df_location.set_index('time').reindex(df_gyro.index, method='nearest')
or use interpolation: method='linear', 'cubic', etc.
Downsampling (high → low frequency):
python# Average high-freq sensors to GPS timestamps
df_gyro_downsampled = df_gyro.groupby(pd.cut(df_gyro['time'], bins=df_location['time'])).mean()
### 3. Common F1 Approach: Multi-Rate Fusion
Create separate analysis streams:

10 Hz stream: Position, speed, gear, throttle, brake (lap analysis, racing line)
100 Hz stream: Gyro, accel, g-forces (corner entry/exit, balance analysis)
Event-based: Gear changes, flags, incidents

### 4. Recommended Structure
python# Primary telemetry at GPS rate (1-10 Hz)
telemetry_df = location_data.merge(
    gyro_data.resample('100ms').mean(),  # downsample gyro
    orientation_data,
    on='time', 
    how='left'
)

High-frequency dynamics (100 Hz) - separate analysis
dynamics_df = gyro + accelerometer + gravity (keep native rate)
### 5. Key Metrics to Calculate

Lateral/Longitudinal G-forces (from accelerometer)
Yaw rate (from gyroscope Z-axis)
Speed/Distance (from GPS)
Cornering zones (high yaw rate + low speed)
Lap segmentation (using GPS coordinates)

Best practice: Keep raw 100Hz data separate for detailed analysis, create downsampled dataset for lap-by-lap comparisons.



## Sampling frequencies

Output the sampling frequencies for each table and then updated based on strategies



In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('./data/karting_merged.sqlite')

# Get all table names
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

print("=== Table Sampling Frequencies ===\n")

# Tables to skip (no time column or metadata)
skip_tables = ['RaceMetadata', 'Tags', 'Metadata', 'Network', 'Battery', 'Brightness']

for table_name in tables['name']:
    if table_name in skip_tables:
        continue
    
    # Get first 100 rows to calculate average frequency
    query = f"SELECT time FROM {table_name} ORDER BY time LIMIT 100"
    try:
        times = pd.read_sql_query(query, conn)
        
        if len(times) > 1:
            # Calculate time differences in nanoseconds
            time_diffs = times['time'].diff().dropna()
            avg_diff_ns = time_diffs.mean()
            
            # Convert to seconds and calculate frequency
            avg_diff_s = avg_diff_ns / 1e9
            frequency_hz = 1 / avg_diff_s
            
            print(f"{table_name}: {frequency_hz:.2f} Hz (avg interval: {avg_diff_s*1000:.2f} ms)")
    except:
        print(f"{table_name}: No time column")

conn.close()

=== Table Sampling Frequencies ===

Gyroscope: 410.19 Hz (avg interval: 2.44 ms)
Pedometer: 1.42 Hz (avg interval: 702.12 ms)
WatchMagnetometer: 99.90 Hz (avg interval: 10.01 ms)
Gravity: 410.19 Hz (avg interval: 2.44 ms)
MagnetometerUncalibrated: 406.44 Hz (avg interval: 2.46 ms)
Magnetometer: 410.19 Hz (avg interval: 2.44 ms)
Location: 4.13 Hz (avg interval: 242.42 ms)
GyroscopeUncalibrated: 410.19 Hz (avg interval: 2.44 ms)
Microphone: 35.56 Hz (avg interval: 28.12 ms)
WatchBarometer: 0.39 Hz (avg interval: 2574.26 ms)
WatchLocation: 0.14 Hz (avg interval: 7009.32 ms)
Orientation: 410.19 Hz (avg interval: 2.44 ms)
AccelerometerUncalibrated: 410.19 Hz (avg interval: 2.44 ms)
Compass: 410.19 Hz (avg interval: 2.44 ms)
Barometer: 3.98 Hz (avg interval: 251.01 ms)
Accelerometer: 410.19 Hz (avg interval: 2.44 ms)
WristMotion: 99.90 Hz (avg interval: 10.01 ms)
Bluetooth: 31.29 Hz (avg interval: 31.96 ms)
BluetoothMetadata: 93.75 Hz (avg interval: 10.67 ms)


## Filter based on start finish

- For each race, remove all 

In [7]:
import sqlite3
import pandas as pd
import json
from shapely import wkt
from shapely.geometry import LineString, Point
import os

# --- Configuration ---
# NEW: Output database path is now ./data/races.sqlite
DB_PATH = './data/races.sqlite'
# Original source table name remains 'Location'
LOCATION_TABLE = 'Location'
# NEW: Filtered output table name is now 'Location' inside the new DB
FILTERED_TABLE = 'Location'
# Coordinates are (Longitude, Latitude) for Shapely, but database stores (Latitude, Longitude)
# We will use 'Longitude' and 'Latitude' columns from the database.

def filter_location_data(db_path, race_config):
    """
    Filters the Location table for each race, keeping only records between 
    the first and last Start/Finish line crossing.
    
    Args:
        db_path (str): Path to the SQLite database.
        race_config (list): List of dictionaries containing race metadata and line data.
    """
    
    # 1. Configuration is already loaded into 'race_config' variable
    config = race_config

    # 2. Database Connection (Connects to the new races.sqlite file)
    try:
        conn = sqlite3.connect(db_path)
        
        # Drop the filtered table (named 'Location' in the new DB) to ensure a clean run
        conn.execute(f"DROP TABLE IF EXISTS {FILTERED_TABLE}")
        conn.commit()
        print(f"New target table '{FILTERED_TABLE}' in '{db_path}' prepared.")
        
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return

    all_filtered_dfs = []

    print("\n--- Starting Data Processing ---")
    
    # 3. We must load data from the original DB path (karting_merged.sqlite)
    # temporarily to perform the filtering.
    ORIGINAL_DB_PATH = './data/karting_merged.sqlite'
    
    # Create a separate connection for reading the source data
    conn_source = sqlite3.connect(ORIGINAL_DB_PATH)

    for entry in config:
        visit = entry['visit']
        race = entry['race']
        
        print(f"Processing Visit {visit}, Race {race}...")

        # 4. Load Race Data from the original source table
        query = f"""
        SELECT * FROM {LOCATION_TABLE}
        WHERE visit = {visit} AND race = {race}
        ORDER BY time
        """
        # Read from the source database connection
        df_race = pd.read_sql_query(query, conn_source)
        
        if df_race.empty:
            print(f"  Warning: No data found for Visit {visit}, Race {race}. Skipping.")
            continue
            
        print(f"  Found {len(df_race)} initial records.")

        # 5. Parse Start/Finish Line
        sf_line = wkt.loads(entry['sf_line'])
            
        # 6. Identify Crossings
        crossing_indices = []
        
        # Iterate through segments (from point i to point i+1)
        for i in range(len(df_race) - 1):
            # Create a LineString for the segment using (Longitude, Latitude) format
            p1_lon, p1_lat = df_race.iloc[i]['Longitude'], df_race.iloc[i]['Latitude']
            p2_lon, p2_lat = df_race.iloc[i+1]['Longitude'], df_race.iloc[i+1]['Latitude']
            
            segment = LineString([(p1_lon, p1_lat), (p2_lon, p2_lat)])
            
            # Check if the segment crosses the sf_line
            if segment.crosses(sf_line):
                # We record the index of the first point of the segment that crosses
                crossing_indices.append(i) 

        # 7. Filter Records
        if not crossing_indices:
            print("  Warning: No Start/Finish line crossings detected. Skipping race.")
            continue
        
        # Start at the point *after* the first recorded crossing segment.
        start_index = crossing_indices[0] + 1
        
        # End at the point *after* the last recorded crossing segment (inclusive).
        end_index = crossing_indices[-1] + 1 
        
        # Filter the DataFrame using slicing
        df_filtered = df_race.iloc[start_index : end_index + 1].copy()
        
        print(f"  Found {len(crossing_indices)} S/F line crossings.")
        print(f"  Keeping records from index {start_index} to {end_index}.")
        print(f"  Resulting filtered records: {len(df_filtered)}")
        
        all_filtered_dfs.append(df_filtered)

    # 8. Write All Filtered Data to New Table
    conn_source.close() # Close connection to the source database

    if all_filtered_dfs:
        # Concatenating preserves the column names and order from the SELECT * query
        final_df = pd.concat(all_filtered_dfs, ignore_index=True)
        # Write to the new database file (DB_PATH = './data/races.sqlite') 
        # into the target table (FILTERED_TABLE = 'Location')
        final_df.to_sql(FILTERED_TABLE, conn, if_exists='replace', index=False)
        print(f"\nSuccessfully wrote {len(final_df)} total filtered records to the '{DB_PATH}' file in the table '{FILTERED_TABLE}'.")
    else:
        print("\nNo data was filtered or written.")
        
    conn.close() # Close connection to the destination database


SyntaxError: no binding for nonlocal 'sector_number' found (642450038.py, line 386)